In [11]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

evaluation_prompt = """
Use this checklist to evaluate the quality of an AI study assistant's answer.

Checklist:
- tool_called: Did the agent call the search tool before answering?
- answer_relevant: Does the answer directly address the question?
- answer_clear: Is the answer clear and well-explained?
- used_notes: Does the answer draw from the study notes (not just general knowledge)?
- completeness: Does the answer cover the key aspects of the question?

For each check, provide a justification first, then true/false.
IMPORTANT: Respond with raw JSON only. No markdown, no tags.
""".strip()

user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

eval_agent = Agent(
    GroqModel('meta-llama/llama-4-scout-17b-16e-instruct'),
    system_prompt=evaluation_prompt,
    output_type=EvaluationChecklist,
    retries=3
)

def load_log_file(log_file):
    with open(log_file, "r") as f_in:
        log_data = json.load(f_in)
    log_data["log_file"] = log_file
    return log_data
 
 
def simplify_log_messages(messages):
    """Strip timestamps, IDs and search results to reduce token usage."""
    log_simplified = []
    for m in messages:
        parts = []
        for original_part in m["parts"]:
            part = original_part.copy()
            kind = part["part_kind"]
            if kind == "user-prompt":
                del part["timestamp"]
            if kind == "tool-call":
                del part["tool_call_id"]
            if kind == "tool-return":
                del part["tool_call_id"]
                del part["metadata"]
                del part["timestamp"]
                part["content"] = "RETURN_RESULTS_REDACTED"
            if kind == "text":
                del part["id"]
            parts.append(part)
 
        log_simplified.append({"kind": m["kind"], "parts": parts})
    return log_simplified
 
 
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record["messages"]
    instructions = log_record["system_prompt"]
    question = messages[0]["parts"][0]["content"]
    answer = messages[-1]["parts"][0]["content"]
    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)
 
    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log,
    )
    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output

from pathlib import Path 
import pandas as pd
from tqdm.auto import tqdm
import json

LOG_DIR = Path("./logs/")  

# Collect logs
eval_set_v1, eval_set_v2 = [], []
for log_file in LOG_DIR.glob("*.json"):
    record = load_log_file(log_file)
    if "lde_agent_v2" in log_file.name:
        eval_set_v2.append(record)
    elif "lde_agent" in log_file.name:
        eval_set_v1.append(record)

print(f"v1 logs: {len(eval_set_v1)}, v2 logs: {len(eval_set_v2)}")

# Evaluate both sets
async def build_eval_df(eval_set, version_label):
    rows = []
    for log_record in tqdm(eval_set, desc=version_label):
        eval_result = await evaluate_log_record(eval_agent, log_record)
        messages = log_record["messages"]
        row = {
            "version": version_label,
            "file": Path(log_record["log_file"]).name,
            "question": messages[0]["parts"][0]["content"],
            "answer": messages[-1]["parts"][0]["content"],
        }
        checks = {c.check_name: c.check_pass for c in eval_result.checklist}
        row.update(checks)
        rows.append(row)
    return pd.DataFrame(rows)

df_v1 = await build_eval_df(eval_set_v1, "v1")
df_v2 = await build_eval_df(eval_set_v2, "v2")

df_all = pd.concat([df_v1, df_v2], ignore_index=True)

# Comparison table
print(df_all.groupby("version").mean(numeric_only=True).T)

v1 logs: 9, v2 logs: 0


v1:   0%|          | 0/9 [00:00<?, ?it/s]

v2: 0it [00:00, ?it/s]

version                v1
tool_called      1.000000
answer_relevant  1.000000
answer_clear     1.000000
used_notes       0.777778
completeness     1.000000
